# AION-U1 – reproduzierbarer LoRA-Pilot

Nur in einer ausdrücklich ausgewählten kostenlosen Colab-GPU-Sitzung ausführen. Dieses Notebook reserviert keine Cloud-Ressourcen und verwendet keine Teacher-API. Die GPU-Verfügbarkeit und der Tarif müssen vom Betreiber geprüft werden. Zellen einzeln ausführen, nicht automatisch alle starten. Der Pilot ist kein Nachweis der 95-%-Zielerreichung.

In [ ]:
# 1. Reproduzierbare Codebasis; keine Trainingsausführung.
import os, pathlib, subprocess, sys
from pathlib import Path
REPO_REVISION = "595dba980a0b935b7ec1f36b9b8160666d3f4e4a"
ROOT = Path("/content/aion-u1-reproducible")
def run(*args):
    subprocess.run(list(map(str, args)), cwd=ROOT, check=True)
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/Reznap87/aion-u1.git", str(ROOT)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True).strip():
    raise RuntimeError("Repository is not clean; use a fresh runtime")
run("git", "checkout", "--detach", REPO_REVISION)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip() == REPO_REVISION
run(sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]")


In [ ]:
# 2. Offline readiness and GPU check; still no model download or training.
import json, torch, yaml
CONFIG = "training/configs/sft_lora_open_v4.yaml"
SUITE = "evaluations/capability_suite/benchmark_hard_v2.yaml"
CAPS = "configs/capabilities.yaml"
cfg = yaml.safe_load((ROOT / CONFIG).read_text())
BASE = cfg["model"]["base_model"]
REVISION = cfg["model"]["revision"]
OUTPUT = ROOT / cfg["run"]["output_dir"]
run("aion-u1-preflight", "--config", CONFIG, "--suite", SUITE, "--capabilities", CAPS)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU available. Do not silently fall back to CPU.")
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
assert cfg["model"]["dtype"] == "float32" and not cfg["training"]["fp16"] and not cfg["training"]["bf16"]


In [ ]:
# 3. Explicit manual authorization; default is blocked.
CONFIRM_FREE_RUNTIME = False  # set True only after checking the session tariff
if CONFIRM_FREE_RUNTIME is not True:
    raise RuntimeError("Confirm that this runtime is free before starting training")
run("aion-u1-train-sft", "--config", CONFIG, "--suite", SUITE)
manifest = json.loads((OUTPUT / "run_manifest.json").read_text())
assert manifest["status"] == "completed"
assert manifest["model"]["resolved_revision"] == REVISION
ADAPTER = OUTPUT / "final_model"
from safetensors.torch import load_file
weights = load_file(str(ADAPTER / "adapter_model.safetensors"))
assert weights and all(bool(torch.isfinite(v).all()) for v in weights.values()), "Non-finite adapter weights"
b_weights = [v for k, v in weights.items() if k.endswith("lora_B.weight")]
assert b_weights and any(bool(v.ne(0).any()) for v in b_weights), "LoRA B stayed zero"
print("Adapter saved; nonzero finite weights verified. Quality is not yet established.")


In [ ]:
# 4. Save the adapter before the runtime expires; no cloud upload.
import shutil
from google.colab import files
archive = shutil.make_archive("/content/aion-u1-pilot-adapter", "zip", root_dir=OUTPUT)
files.download(archive)


## Optional evaluation

The v2 benchmark needs an isolated Docker runtime for functional code checks. Colab may not provide Docker; missing checks must remain unmeasured. Image evaluation is also currently unsupported. Reports are therefore potentially incomplete and cannot establish the full AION-Index. Baseline and adapter must use the same suite, revision and token budget. No historical index threshold is assumed.

In [ ]:
# 5. Optional: evaluate both variants with the same setup.
CONFIRM_EVALUATION = False
if CONFIRM_FREE_RUNTIME is not True or CONFIRM_EVALUATION is not True:
    raise RuntimeError("Explicitly enable optional evaluation on the free runtime")
common = ["aion-u1-evaluate-model", "--base-model", BASE, "--revision", REVISION,
          "--suite", SUITE, "--device", "cuda", "--max-tokens", "1024"]
run(*common, "--output", "reports/eval-baseline-v2.json")
run(*common, "--adapter", ADAPTER, "--output", "reports/eval-pilot-v2.json")
for name in ("eval-baseline-v2", "eval-pilot-v2"):
    d = json.loads((ROOT / "reports" / (name + ".json")).read_text())
    s = d["scores"]["lokal"]
    print(name, "index", s["overall_score"], "measured", s["measured_score"], "coverage", s["coverage"], "errors", s["errored"])


In [ ]:
# 6. Download both evaluation reports locally.
import zipfile
report_archive = "/content/aion-u1-pilot-evaluations.zip"
with zipfile.ZipFile(report_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for name in ("eval-baseline-v2.json", "eval-pilot-v2.json"):
        bundle.write(ROOT / "reports" / name, arcname=name)
files.download(report_archive)
